In [0]:
%python
import re

# =========================================
# CONFIGURATION
# =========================================

base_path = "s3://my-retail-lakehouse"

zones = [
    "bronze",
    "processed"
]

entities = [
    "customers",
    "products",
    "stores",
    "sales"
]

# =========================================
# FILE NAMING CONVENTION
# Example:
# customers_06052026_101500.csv
# =========================================

pattern = r'.*_\d{14}\.csv$'

# =========================================
# START ARCHIVAL PROCESS
# =========================================

for zone in zones:

    print(f"\n========== PROCESSING ZONE: {zone.upper()} ==========")

    for entity in entities:

        source_path = f"{base_path}/{zone}/{entity}/"

        archive_path = f"{base_path}/archive/{zone}/{entity}/"

        print(f"\nChecking path: {source_path}")

        try:

            # =========================================
            # READ FILES
            # =========================================

            files = dbutils.fs.ls(source_path)

            # =========================================
            # FILTER VALID FILES
            # =========================================

            valid_files = []

            for file in files:

                if re.match(pattern, file.name):

                    valid_files.append(file)

            # =========================================
            # VALIDATION: FILES EXIST
            # =========================================

            if len(valid_files) == 0:

                print("No valid files found")

                continue

            # =========================================
            # SORT FILES
            # Latest timestamp file last
            # =========================================

            if entity == "products":
                # Sort by timestamp extracted from filename for products
                def extract_timestamp(filename):
                    match = re.search(r'_(\d{14})\.csv$', filename)
                    return match.group(1) if match else ""
                valid_files = sorted(
                    valid_files,
                    key=lambda x: extract_timestamp(x.name)
                )
            else:
                valid_files = sorted(
                    valid_files,
                    key=lambda x: x.name
                )

            latest_file = valid_files[-1]

            old_files = valid_files[:-1]

            print(f"Latest active file retained: {latest_file.name}")

            # =========================================
            # MOVE OLD FILES TO ARCHIVE
            # =========================================

            for old_file in old_files:

                destination = archive_path + old_file.name

                dbutils.fs.mv(
                    old_file.path,
                    destination
                )

            print(f"Archival completed for {entity}")

        except Exception as e:

            print(f"Error processing {entity}: {str(e)}")

print("\n===================================\nARCHIVAL PROCESS COMPLETED\n===================================")

In [0]:
%python

import re

# =========================================
# CONFIGURATION
# =========================================

base_path = "s3://my-retail-lakehouse"

zones = [
    "bronze",
    "processed"
]

entities = [
    "customers",
    "products",
    "stores",
    "sales"
]

# =========================================
# FILE NAME VALIDATION PATTERN
# Example:
# customers_src_20042026100105.csv
# =========================================

pattern = r'.*_\d{14}\.csv$'


for zone in zones:
    print(f"\n========== VALIDATING ZONE: {zone.upper()} ==========")
    for entity in entities:
        source_path = f"{base_path}/{zone}/{entity}/"
        archive_path = f"{base_path}/archive/{zone}/{entity}/"
        print(f"\nChecking entity: {entity}")
        try:
            active_files = dbutils.fs.ls(source_path)
            valid_active_files = []
            for file in active_files:
                if re.match(pattern, file.name):
                    valid_active_files.append(file.name)
            try:
                archived_files = dbutils.fs.ls(archive_path)
                valid_archived_files = []
                for file in archived_files:
                    if re.match(pattern, file.name):
                        valid_archived_files.append(file.name)
            except:
                valid_archived_files = []
            if len(valid_active_files) == 1:
                print("PASS: Only latest active file exists")
            elif len(valid_active_files) == 0:
                print("WARNING: No active files found")
            else:
                print("FAIL: Multiple active files found")
            if len(valid_archived_files) > 0:
                print("PASS: Archived files detected")
            else:
                print("INFO: No archived files yet")
            invalid_files = []
            for file in active_files:
                if not re.match(pattern, file.name):
                    invalid_files.append(file.name)
            if len(invalid_files) == 0:
                print("PASS: File naming convention valid")
            else:
                print("FAIL: Invalid file names detected")
                for invalid in invalid_files:
                    print(f"Invalid file: {invalid}")
            print("\nActive Files:")
            for file in valid_active_files:
                print(file)
            print("\nArchived Files:")
            for file in valid_archived_files:
                print(file)
        except Exception as e:
            print(f"ERROR validating {entity}")
            print(str(e))
print("\n===================================")
print("ARCHIVAL VALIDATION COMPLETED")
print("===================================")